<a href="https://colab.research.google.com/github/zFonta/CEIA/blob/main/AMIA/AMIA_2025_TP1_V2Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP 1: LDA/QDA y optimización matemática de modelos

## Alumnos:
 -Juan Sebastián Bonal, jsbonals@gmail.com <br>
 -Federico Santiago Fontanari, federicofontanari@gmail.com <br>
 -Jose Andres Montes de Oca, amontesdeoca1982@gmail.com <br>

# Intro teórica

## Definición: Clasificador Bayesiano

Sean $k$ poblaciones, $x \in \mathbb{R}^p$ puede pertenecer a cualquiera $g \in \mathcal{G}$ de ellas. Bajo un esquema bayesiano, se define entonces $\pi_j \doteq P(G = j)$ la probabilidad *a priori* de que $X$ pertenezca a la clase *j*, y se **asume conocida** la distribución condicional de cada observable dado su clase $f_j \doteq f_{X|G=j}$.

De esta manera dicha probabilidad *a posteriori* resulta
$$
P(G|_{X=x} = j) = \frac{f_{X|G=j}(x) \cdot p_G(j)}{f_X(x)} \propto f_j(x) \cdot \pi_j
$$

La regla de decisión de Bayes es entonces
$$
H(x) \doteq \arg \max_{g \in \mathcal{G}} \{ P(G|_{X=x} = j) \} = \arg \max_{g \in \mathcal{G}} \{ f_j(x) \cdot \pi_j \}
$$

es decir, se predice a $x$ como perteneciente a la población $j$ cuya probabilidad a posteriori es máxima.

*Ojo, a no desesperar! $\pi_j$ no es otra cosa que una constante prefijada, y $f_j$ es, en su esencia, un campo escalar de $x$ a simplemente evaluar.*

## Distribución condicional

Para los clasificadores de discriminante cuadrático y lineal (QDA/LDA) se asume que $X|_{G=j} \sim \mathcal{N}_p(\mu_j, \Sigma_j)$, es decir, se asume que cada población sigue una distribución normal.

Por definición, se tiene entonces que para una clase $j$:
$$
f_j(x) = \frac{1}{(2 \pi)^\frac{p}{2} \cdot |\Sigma_j|^\frac{1}{2}} e^{- \frac{1}{2}(x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j)}
$$

Aplicando logaritmo (que al ser una función estrictamente creciente no afecta el cálculo de máximos/mínimos), queda algo mucho más práctico de trabajar:

$$
\log{f_j(x)} = -\frac{1}{2}\log |\Sigma_j| - \frac{1}{2} (x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j) + C
$$

Observar que en este caso $C=-\frac{p}{2} \log(2\pi)$, pero no se tiene en cuenta ya que al tener una constante aditiva en todas las clases, no afecta al cálculo del máximo.

## LDA

En el caso de LDA se hace una suposición extra, que es $X|_{G=j} \sim \mathcal{N}_p(\mu_j, \Sigma)$, es decir que las poblaciones no sólo siguen una distribución normal sino que son de igual matriz de covarianzas. Reemplazando arriba se obtiene entonces:

$$
\log{f_j(x)} =  -\frac{1}{2}\log |\Sigma| - \frac{1}{2} (x-\mu_j)^T \Sigma^{-1} (x- \mu_j) + C
$$

Ahora, como $-\frac{1}{2}\log |\Sigma|$ es común a todas las clases se puede incorporar a la constante aditiva y, distribuyendo y reagrupando términos sobre $(x-\mu_j)^T \Sigma^{-1} (x- \mu_j)$ se obtiene finalmente:

$$
\log{f_j(x)} =  \mu_j^T \Sigma^{-1} (x- \frac{1}{2} \mu_j) + C'
$$

## Entrenamiento/Ajuste

Obsérvese que para ambos modelos, ajustarlos a los datos implica estimar los parámetros $(\mu_j, \Sigma_j) \; \forall j = 1, \dots, k$ en el caso de QDA, y $(\mu_j, \Sigma)$ para LDA.

Estos parámetros se estiman por máxima verosimilitud, de manera que los estimadores resultan:

* $\hat{\mu}_j = \bar{x}_j$ el promedio de los $x$ de la clase *j*
* $\hat{\Sigma}_j = s^2_j$ la matriz de covarianzas estimada para cada clase *j*
* $\hat{\pi}_j = f_{R_j} = \frac{n_j}{n}$ la frecuencia relativa de la clase *j* en la muestra
* $\hat{\Sigma} = \frac{1}{n} \sum_{j=1}^k n_j \cdot s^2_j$ el promedio ponderado (por frecs. relativas) de las matrices de covarianzas de todas las clases. *Observar que se utiliza el estimador de MV y no el insesgado*

Es importante notar que si bien todos los $\mu, \Sigma$ deben ser estimados, la distribución *a priori* puede no inferirse de los datos sino asumirse previamente, utilizándose como entrada del modelo.

## Predicción

Para estos modelos, al igual que para cualquier clasificador Bayesiano del tipo antes visto, la estimación de la clase es por método *plug-in* sobre la regla de decisión $H(x)$, es decir devolver la clase que maximiza $\hat{f}_j(x) \cdot \hat{\pi}_j$, o lo que es lo mismo $\log\hat{f}_j(x) + \log\hat{\pi}_j$.

# Código provisto

Con el fin de no retrasar al alumno con cuestiones estructurales y/o secundarias al tema que se pretende tratar, se provee una base de código que **no es obligatoria de usar** pero se asume que resulta resulta beneficiosa.

In [ ]:
import numpy as np
import pandas as pd
import numpy.linalg as LA
from scipy.linalg import cholesky, solve_triangular
from scipy.linalg.lapack import dtrtri

## Base code

In [ ]:
class BaseBayesianClassifier:
  def __init__(self):
    pass

  def _estimate_a_priori(self, y):
    a_priori = np.bincount(y.flatten().astype(int)) / y.size
    # Q3: para que sirve bincount?
    return np.log(a_priori)

  def _fit_params(self, X, y):
    # estimate all needed parameters for given model
    raise NotImplementedError()

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    raise NotImplementedError()

  def fit(self, X, y, a_priori=None):
    # if it's needed, estimate a priori probabilities
    self.log_a_priori = self._estimate_a_priori(y) if a_priori is None else np.log(a_priori)

    # now that everything else is in place, estimate all needed parameters for given model
    self._fit_params(X, y)
    # Q4: por que el _fit_params va al final? no se puede mover a, por ejemplo, antes de la priori?

  def predict(self, X):
    # this is actually an individual prediction encased in a for-loop
    m_obs = X.shape[1]
    y_hat = np.empty(m_obs, dtype=int)

    for i in range(m_obs):
      y_hat[i] = self._predict_one(X[:,i].reshape(-1,1))

    # return prediction as a row vector (matching y)
    return y_hat.reshape(1,-1)

  def _predict_one(self, x):
    # calculate all log posteriori probabilities (actually, +C)
    log_posteriori = [ log_a_priori_i + self._predict_log_conditional(x, idx) for idx, log_a_priori_i
                  in enumerate(self.log_a_priori) ]

    # return the class that has maximum a posteriori probability
    return np.argmax(log_posteriori)

In [ ]:
class QDA(BaseBayesianClassifier):

  def _fit_params(self, X, y):
    # estimate each covariance matrix
    self.inv_covs = [LA.inv(np.cov(X[:,y.flatten()==idx], bias=True))
                      for idx in range(len(self.log_a_priori))]
    # Q5: por que hace falta el flatten y no se puede directamente X[:,y==idx]?
    # Q6: por que se usa bias=True en vez del default bias=False?
    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]
    # Q7: que hace axis=1? por que no axis=0?

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    inv_cov = self.inv_covs[class_idx]
    unbiased_x =  x - self.means[class_idx]
    return 0.5*np.log(LA.det(inv_cov)) -0.5 * unbiased_x.T @ inv_cov @ unbiased_x

In [ ]:
class TensorizedQDA(QDA):

    def _fit_params(self, X, y):
        # ask plain QDA to fit params
        super()._fit_params(X,y)

        # stack onto new dimension
        self.tensor_inv_cov = np.stack(self.inv_covs)
        self.tensor_means = np.stack(self.means)

    def _predict_log_conditionals(self,x):
        unbiased_x = x - self.tensor_means
        inner_prod = unbiased_x.transpose(0,2,1) @ self.tensor_inv_cov @ unbiased_x

        return 0.5*np.log(LA.det(self.tensor_inv_cov)) - 0.5 * inner_prod.flatten()

    def _predict_one(self, x):
        # return the class that has maximum a posteriori probability
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

In [ ]:
class QDA_Chol1(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.L_invs = [
        LA.inv(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True))
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = L_inv @ unbiased_x

    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()

In [ ]:
class QDA_Chol2(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.Ls = [
        cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True)
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L = self.Ls[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = solve_triangular(L, unbiased_x, lower=True)

    return -np.log(L.diagonal().prod()) -0.5 * (y**2).sum()

In [ ]:
class QDA_Chol3(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.L_invs = [
        dtrtri(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True), lower=1)[0]
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = L_inv @ unbiased_x

    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()

## Datasets

Observar que se proveen **4 datasets diferentes**, el código de ejemplo usa uno solo pero eso no significa que ustedes se limiten al mismo. También pueden usar otros datasets de su elección.

In [ ]:
from sklearn.datasets import load_iris, fetch_openml, load_wine
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

def get_iris_dataset():
  data = load_iris()
  X_full = data.data
  y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
  return X_full, y_full

def get_penguins_dataset():
    # get data
    df, tgt = fetch_openml(name="penguins", return_X_y=True, as_frame=True, parser='auto')

    # drop non-numeric columns
    df.drop(columns=["island","sex"], inplace=True)

    # drop rows with missing values
    mask = df.isna().sum(axis=1) == 0
    df = df[mask]
    tgt = tgt[mask]

    return df.values, tgt.to_numpy().reshape(-1,1)

def get_wine_dataset():
    # get data
    data = load_wine()
    X_full = data.data
    y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
    return X_full, y_full

def get_letters_dataset():
    # get data
    letter = fetch_openml('letter', version=1, as_frame=False)
    return letter.data, letter.target.reshape(-1,1)

def label_encode(y_full):
    return LabelEncoder().fit_transform(y_full.flatten()).reshape(y_full.shape)

def split_transpose(X, y, test_size, random_state):
    # X_train, X_test, y_train, y_test but all transposed
    return [elem.T for elem in train_test_split(X, y, test_size=test_size, random_state=random_state)]

## Benchmarking

Nota: esta clase fue creada bastante rápido y no pretende ser una plataforma súper confiable sobre la que basarse, sino más bien una herramienta simple con la que poder medir varios runs y agregar la información.

En forma rápida, `warmup` es la cantidad de runs para warmup, `mem_runs` es la cantidad de runs en las que se mide el pico de uso de RAM y `n_runs` es la cantidad de runs en las que se miden tiempos.

La razón por la que se separan es que medir memoria hace ~2.5x más lento cada run, pero al mismo tiempo se estabiliza mucho más rápido.

**Importante:** tener en cuenta que los modelos que predicen en batch (usan `predict` directamente) deberían consumir, como mínimo, $n$ veces la memoria de los que predicen por observación.

In [ ]:
import time
from tqdm.notebook import tqdm
from numpy.random import RandomState
import tracemalloc

RNG_SEED = 6553

class Benchmark:
    def __init__(self, X, y, n_runs=1000, warmup=100, mem_runs=100, test_sz=0.3, rng_seed=RNG_SEED, same_splits=True):
        self.X = X
        self.y = y
        self.n = n_runs
        self.warmup = warmup
        self.mem_runs = mem_runs
        self.test_sz = test_sz
        self.det = same_splits
        if self.det:
            self.rng_seed = rng_seed
        else:
            self.rng = RandomState(rng_seed)

        self.data = dict()

        print("Benching params:")
        print("Total runs:",self.warmup+self.mem_runs+self.n)
        print("Warmup runs:",self.warmup)
        print("Peak Memory usage runs:", self.mem_runs)
        print("Running time runs:", self.n)
        approx_test_sz = int(self.y.size * self.test_sz)
        print("Train size rows (approx):",self.y.size - approx_test_sz)
        print("Test size rows (approx):",approx_test_sz)
        print("Test size fraction:",self.test_sz)

    def bench(self, model_class, **kwargs):
        name = model_class.__name__
        time_data = np.empty((self.n, 3), dtype=float)  # train_time, test_time, accuracy
        mem_data = np.empty((self.mem_runs, 2), dtype=float)  # train_peak_mem, test_peak_mem
        rng = RandomState(self.rng_seed) if self.det else self.rng


        for i in range(self.warmup):
            # Instantiate model with error check for unsupported parameters
            model = model_class(**kwargs)

            # Generate current train-test split
            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )
            # Run training and prediction (timing or memory measurement not recorded)
            model.fit(X_train, y_train)
            model.predict(X_test)

        for i in tqdm(range(self.mem_runs), total=self.mem_runs, desc=f"{name} (MEM)"):

            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            tracemalloc.start()

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()

            _, train_peak = tracemalloc.get_traced_memory()
            tracemalloc.reset_peak()

            model.predict(X_test)
            t3 = time.perf_counter()
            _, test_peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            mem_data[i,] = (
                train_peak / (1024 * 1024),
                test_peak / (1024 * 1024)
            )

        for i in tqdm(range(self.n), total=self.n, desc=f"{name} (TIME)"):
            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()
            preds = model.predict(X_test)
            t3 = time.perf_counter()

            time_data[i,] = (
                (t2 - t1) * 1000,
                (t3 - t2) * 1000,
                (y_test.flatten() == preds.flatten()).mean()
            )

        self.data[name] = (time_data, mem_data)

    def summary(self, baseline=None):
        aux = []
        for name, (time_data, mem_data) in self.data.items():
            result = {
                'model': name,
                'train_median_ms': np.median(time_data[:, 0]),
                'train_std_ms': time_data[:, 0].std(),
                'test_median_ms': np.median(time_data[:, 1]),
                'test_std_ms': time_data[:, 1].std(),
                'mean_accuracy': time_data[:, 2].mean(),
                'train_mem_median_mb': np.median(mem_data[:, 0]),
                'train_mem_std_mb': mem_data[:, 0].std(),
                'test_mem_median_mb': np.median(mem_data[:, 1]),
                'test_mem_std_mb': mem_data[:, 1].std()
            }
            aux.append(result)
        df = pd.DataFrame(aux).set_index('model')

        if baseline is not None and baseline in self.data:
            df['train_speedup'] = df.loc[baseline, 'train_median_ms'] / df['train_median_ms']
            df['test_speedup'] = df.loc[baseline, 'test_median_ms'] / df['test_median_ms']
            df['train_mem_reduction'] = df.loc[baseline, 'train_mem_median_mb'] / df['train_mem_median_mb']
            df['test_mem_reduction'] = df.loc[baseline, 'test_mem_median_mb'] / df['test_mem_median_mb']
        return df

## Ejemplo

In [ ]:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()

X_full.shape, y_full.shape

((178, 13), (178, 1))

In [ ]:
# encodeamos a número las clases
y_full_encoded = label_encode(y_full)

y_full[:5], y_full_encoded[:5]

(array([['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0']], dtype='<U7'),
 array([[0],
        [0],
        [0],
        [0],
        [0]]))

In [ ]:
# generamos el benchmark
# observar que son valores muy bajos de runs para que corra rápido ahora
b = Benchmark(
    X_full, y_full_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


In [ ]:
# bencheamos un par
to_bench = [QDA]

for model in to_bench:
    b.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
# como es una clase, podemos seguir bencheando más después
b.bench(TensorizedQDA)

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
# hacemos un summary
b.summary()

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb
model,,,,,,,,,
QDA,0.4456,0.278530,3.20515,0.543599,0.982407,0.018520,0.042177,0.007627,0.042192
TensorizedQDA,0.4072,0.228501,1.27540,0.349832,0.982593,0.018372,0.042197,0.012001,0.042293


In [ ]:
# son muchos datos! nos quedamos con un par nomás
summ = b.summary()

# como es un pandas DataFrame, subseteamos columnas fácil
summ[['train_median_ms', 'test_median_ms','mean_accuracy']]

,train_median_ms,test_median_ms,mean_accuracy
model,,,
QDA,0.4456,3.20515,0.982407
TensorizedQDA,0.4072,1.27540,0.982593


In [ ]:
# podemos setear un baseline para que fabrique columnas de comparación
summ = b.summary(baseline='QDA')

summ

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,0.4456,0.278530,3.20515,0.543599,0.982407,0.018520,0.042177,0.007627,0.042192,1.000000,1.000000,1.000000,1.00000
TensorizedQDA,0.4072,0.228501,1.27540,0.349832,0.982593,0.018372,0.042197,0.012001,0.042293,1.094302,2.513055,1.008098,0.63549


In [ ]:
summ[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.4456,3.20515,0.982407,1.000000,1.000000,1.000000,1.00000
TensorizedQDA,0.4072,1.27540,0.982593,1.094302,2.513055,1.008098,0.63549


# Consigna QDA

**Notación**: en general notamos

* $k$ la cantidad de clases
* $n$ la cantidad de observaciones
* $p$ la cantidad de features/variables/predictores

**Sugerencia:** combinaciones adecuadas de `transpose`, `stack`, `reshape` y, ocasionalmente, `flatten` y `diagonal` suele ser más que suficiente. Se recomienda *fuertemente* explorar la dimensionalidad de cada elemento antes de implementar las clases.

## Tensorización

En esta sección nos vamos a ocupar de hacer que el modelo sea más rápido para generar predicciones, observando que incurre en un doble `for` dado que predice en forma individual un escalar para cada observación, para cada clase. Paralelizar ambos vía tensorización suena como una gran vía de mejora de tiempos.

### 1) Diferencias entre `QDA`y `TensorizedQDA`

1. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?
2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.

### 2) Optimización

Debido a la forma cuadrática de QDA, no se puede predecir para $n$ observaciones en una sola pasada (utilizar $X \in \mathbb{R}^{p \times n}$ en vez de $x \in \mathbb{R}^p$) sin pasar por una matriz de $n \times n$ en donde se computan todas las interacciones entre observaciones. Se puede acceder al resultado recuperando sólo la diagonal de dicha matriz, pero resulta ineficiente en tiempo y (especialmente) en memoria. Aún así, es *posible* que el modelo funcione más rápido.

3. Implementar el modelo `FasterQDA` (se recomienda heredarlo de `TensorizedQDA`) de manera de eliminar el ciclo for en el método predict.
4. Mostrar dónde aparece la mencionada matriz de $n \times n$, donde $n$ es la cantidad de observaciones a predecir.
5. Demostrar que
$$
diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)
$$ es decir, que se puede "esquivar" la matriz de $n \times n$ usando matrices de $n \times p$. También se puede usar, de forma equivalente,
$$
np.sum(A^T \odot B, axis=0).T
$$
queda a preferencia del alumno cuál usar.
6. Utilizar la propiedad antes demostrada para reimplementar la predicción del modelo `FasterQDA` de forma eficiente en un nuevo modelo `EfficientQDA`.
7. Comparar la performance de las 4 variantes de QDA implementadas hasta ahora (no Cholesky) ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

## Cholesky

Hasta ahora todos los esfuerzos fueron enfocados en realizar una predicción más rápida. Los tiempos de entrenamiento (teóricos al menos) siguen siendo los mismos o hasta (minúsculamente) peores, dado que todas las mejoras siguen llamando al método `_fit_params` original de `QDA`.

La descomposición/factorización de [Cholesky](https://en.wikipedia.org/wiki/Cholesky_decomposition#Statement) permite factorizar una matriz definida positiva $A = LL^T$ donde $L$ es una matriz triangular inferior. En particular, si bien se asume que $p \ll n$, invertir la matriz de covarianzas $\Sigma$ para cada clase impone un cuello de botella que podría alivianarse. Teniendo en cuenta que las matrices de covarianza son simétricas y salvo degeneración, definidas positivas, Cholesky como mínimo debería permitir invertir la matriz más rápido.

*Nota: observar que calcular* $A^{-1}b$ *equivale a resolver el sistema* $Ax=b$.

### 3) Diferencias entre implementaciones de `QDA_Chol`

8. Si una matriz $A$ tiene fact. de Cholesky $A=LL^T$, expresar $A^{-1}$ en términos de $L$. ¿Cómo podría esto ser útil en la forma cuadrática de QDA?
7. Explicar las diferencias entre `QDA_Chol1`y `QDA` y cómo `QDA_Chol1` llega, paso a paso, hasta las predicciones.
8. ¿Cuáles son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?
9. Comparar la performance de las 7 variantes de QDA implementadas hasta ahora ¿Qué se observa?¿Hay alguna de las implementaciones de `QDA_Chol` que sea claramente mejor que las demás?¿Alguna que sea peor?

### 4) Optimización

12. Implementar el modelo `TensorizedChol` paralelizando sobre clases/observaciones según corresponda. Se recomienda heredarlo de alguna de las implementaciones de `QDA_Chol`, aunque la elección de cuál de ellas queda a cargo del alumno según lo observado en los benchmarks de puntos anteriores.
13. Implementar el modelo `EfficientChol` combinando los insights de `EfficientQDA` y `TensorizedChol`. Si se desea, se puede implementar `FasterChol` como ayuda, pero no se contempla para el punto.
13. Comparar la performance de las 9 variantes de QDA implementadas ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

## Importante:

Las métricas que se observan al realizar benchmarking son muy dependientes del código que se ejecuta, y por tanto de las versiones de las librerías utilizadas. Una forma de unificar esto es utilizando un gestor de versiones y paquetes como _uv_ o _Poetry_, otra es simplemente usando una misma VM como la que provee Colab.

**Cada equipo debe informar las versiones de Python, NumPy y SciPy con que fueron obtenidos los resultados. En caso de que sean múltiples, agregar todos los casos**. La siguiente celda provee una ayuda para hacerlo desde un notebook, aunque como es una secuencia de comandos también sirve para consola.

In [ ]:
#%%bash
#python --version
#pip freeze | grep -E "scipy|numpy"

**Comentario:** yo utilicé los siguientes parámetros para mi run de prueba. Esto NO significa que ustedes tengan que usar los mismos, tampoco el mismo dataset. Se agregó al notebook simplemente porque fue una pregunta común en cohortes anteriores.

In [ ]:
# dataset de letters
X_letter, y_letter = get_letters_dataset()

# encoding de labels
y_letter_encoded = label_encode(y_letter.reshape(-1,1))

# instanciacion del benchmark
b = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


# Respuestas

Vamos a verificar primero las versiones de numpy y scipy

In [ ]:
import numpy as np
import scipy as sp
import sys

print(f"Python {sys.version}")
print(f"numpy=={np.__version__}")
print(f"scipy=={sp.__version__}")

Python 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
numpy==2.1.3
scipy==1.15.3


Elegimos el dataset de Letters para hacer las pruebas ya que es el mas grande de los 4 disponibles, por lo cual se notara mas si un modelo esta optimizado o no.

In [ ]:
x,y=get_iris_dataset()
print("Shape Iris: "+ str(x.shape))

x,y=get_penguins_dataset()
print("Shape Penguins: "+ str(x.shape))

x,y=get_wine_dataset()
print("Shape Wine: "+ str(x.shape))

x,y=get_letters_dataset()
print("Shape Letters: "+ str(x.shape))


Shape Iris: (150, 4)
Shape Penguins: (342, 4)
Shape Wine: (178, 13)
Shape Letters: (20000, 16)


In [ ]:
X_letter, y_letter = get_letters_dataset()
# encoding de labels
y_letter_encoded = label_encode(y_letter.reshape(-1,1))

# Vamos a hacer train/test manual para evaluar manualmente modelo por modelo
X_train, X_test, y_train, y_test = split_transpose(X_letter, y_letter_encoded, 0.3, 14)
print(X_train.shape)
print(y_train.shape)

(16, 14000)
(1, 14000)


In [ ]:
print("Clases distintas (k): "+str(len(set(y_train[0]))))
print("Predictores (p): "+str(len(X_train)))

Clases distintas (k): 26
Predictores (p): 16


$1$. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?

`TensorizedQDA` paraleliza sobre las $k$ clases.
Si se observa el método `predict` heredado de `BaseBayesianClassifier`, la predicción principal sigue ocurriendo dentro de un ciclo `for i in range(m_obs):` <br> Lo que hace `TensorizedQDA` es optimizar el método `_predict_log_conditionals(x)` (x sigue siendo una sola observacion) para calcular la probabilidad a posteriori de esa observación en todas las clases al mismo tiempo usando tensores

$2$. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.

Siendo $p$ la cantidad de predictores, y $k$ la cantidad de clases:

`tensor_means` apila los $k$ vectores de medias de tamaño $p×1$. Su shape final es $(k,p,1)$

`tensor_inv_cov` apila las $k$ matrices de covarianza inversa de tamaño $p×p$. Su shape final es $(k,p,p)$

In [ ]:
#QDA:
model_QDA=QDA()
model_QDA.fit(X_train, y_train)
preds_QDA=model_QDA.predict(X_test)

#TensorizedQDA
model_TensorizedQDA=TensorizedQDA()
model_TensorizedQDA.fit(X_train, y_train)
preds_TensorizedQDA=model_TensorizedQDA.predict(X_test)

#Comparo
if np.array_equal(preds_QDA, preds_TensorizedQDA):
    print("Ambas predicciones son Iguales")

Ambas predicciones son Iguales


In [ ]:
print("Demostracion Empirica con Dataset Letters, k=26, p=16")
print("Shape de tensor_means: " + str(model_TensorizedQDA.tensor_means.shape))
print("Shape de tensor_inv_cov: " + str(model_TensorizedQDA.tensor_inv_cov.shape))

Demostracion Empirica con Dataset Letters, k=26, p=16
Shape de tensor_means: (26, 16, 1)
Shape de tensor_inv_cov: (26, 16, 16)


Dada una observacion $x$ de shape $(p,1)$:<br>
`QDA` itera sobre las $k$ clases y calcula para cada clase $k$ el logaritmo de la probabilidad condicionada dada esa clase.

`TensorizedQDA`, hace todo esto junto sin necesidad de iterar para cada clase $k$ :<br>
-1.Broadcasting: Resta la media de todas las clases a la vez con `unbiased_x = x - self.tensor_means`. NumPy expande automaticamente $x$ en $k$ clases. El resultado tiene shape $(k,p,1)$.

-2.Producto Interno : El cálculo `unbiased_x.transpose(0,2,1) @ self.tensor_inv_cov @ unbiased_x` realiza multiplicaciones de matrices por lotes. Las dimensiones operadas son $(k,1,p)$ × $(k,p,p)$ × $(k,p,1)$, lo que da como resultado un tensor de shape $(k,1,1)$.

-3.Formato: Al aplicar `.flatten()` transforma el tensor $(k,1,1)$ a un vector plano de shape $(k,)$ donde el i-ésimo elemento corresponde a la clase i.

**En resumen**, `TensorizedQDA` calcula exactamente el mismo valor que `QDA` para cada clase $k$. La diferencia es que en lugar de iterar con un bucle `for` sobre las $k$ clases, apila los parámetros en tensores y ejecuta los productos matriciales todos juntos. El resultado es numéricamente idéntico pero computacionalmente más eficiente.


$3$. Implementar el modelo `FasterQDA` (se recomienda heredarlo de `TensorizedQDA`) de manera de eliminar el ciclo for en el método predict.

In [ ]:
class FasterQDA(TensorizedQDA):
    def predict(self, X):
        # X shape: (p, n). tensor_means shape: (k, p, 1)
        # Añadimos una dimensión a X para que el broadcasting funcione: (1, p, n)
        unbiased_X = X[np.newaxis, :, :] - self.tensor_means # shape: (k, p, n)

        # Producto interno: (k, n, p) @ (k, p, p) @ (k, p, n) -> (k, n, n)
        inner_prod = unbiased_X.transpose(0, 2, 1) @ self.tensor_inv_cov @ unbiased_X

        # Extraemos la diagonal de tamaño (n,) para cada una de las k matrices: (k, n)
        quad_form = np.diagonal(inner_prod, axis1=1, axis2=2)

        log_dets = np.log(LA.det(self.tensor_inv_cov)).reshape(-1, 1)
        log_conds = 0.5 * log_dets - 0.5 * quad_form

        log_posteriori = self.log_a_priori.reshape(-1, 1) + log_conds
        return np.argmax(log_posteriori, axis=0).reshape(1, -1)


In [ ]:
#FasterQDA
model_FasterQDA=FasterQDA()
model_FasterQDA.fit(X_train, y_train)
preds_FasterQDA=model_FasterQDA.predict(X_test)

#Comparo
if np.array_equal(preds_QDA, preds_FasterQDA):
    print("Ambas predicciones son Iguales")

Ambas predicciones son Iguales


$4$. Mostrar dónde aparece la mencionada matriz de $n \times n$, donde $n$ es la cantidad de observaciones a predecir.

Aparece en la variable `inner_prod`. Al multiplicar matrices de shape $(k,n,p)$ x $(k,p,p)$ x $(k,p,n)$, el resultado es un tensor de tamaño $(k,n,n)$

$5$. Demostrar que
$$
diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)
$$ es decir, que se puede "esquivar" la matriz de $n \times n$ usando matrices de $n \times p$. También se puede usar, de forma equivalente,
$$
np.sum(A^T \odot B, axis=0).T
$$
queda a preferencia del alumno cuál usar.

Se pide demostrar que para calcular la diagonal del producto de dos matrices no es necesario calcular el producto matricial completo $A \cdot B$

La identidad a demostrar es:
$$\text{diag}(A \cdot B) = \sum_{cols} (A \odot B^T)$$

Definimos la matriz $A$ de $n$ filas y $p$ columnas, y la matriz $B$ de $p$ filas y $n$ columnas:

$$
A = \begin{pmatrix}
a_{11} & a_{12} & \dots & a_{1p} \\
a_{21} & a_{22} & \dots & a_{2p} \\
\vdots & \vdots & \ddots & \vdots \\
a_{n1} & a_{n2} & \dots & a_{np}
\end{pmatrix}_{n \times p}
\quad
B = \begin{pmatrix}
b_{11} & b_{12} & \dots & b_{1n} \\
b_{21} & b_{22} & \dots & b_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
b_{p1} & b_{p2} & \dots & b_{pn}
\end{pmatrix}_{p \times n}
$$

##### 1. Producto Matricial $C = A \cdot B$
Al multiplicar $A \cdot B$, obtenemos una matriz $C$ de $n \times n$. El elemento $i$-ésimo de su diagonal se obtiene multiplicando la fila $i$ de $A$ por la columna $i$ de $B$

$$C_{ii} = a_{i1}b_{1i} + a_{i2}b_{2i} + \dots + a_{ip}b_{pi} = \sum_{k=1}^{p} a_{ik}b_{ki}$$


##### 2. Transposición de $B$
Para realizar el producto elemento a elemento, necesitamos que $B$ tenga la misma forma que $A$. Al trasponer $B$, las filas se convierten en columnas:

$$
B^T = \begin{pmatrix}
b_{11} & b_{21} & \dots & b_{p1} \\
b_{12} & b_{22} & \dots & b_{p2} \\
\vdots & \vdots & \ddots & \vdots \\
b_{1n} & b_{2n} & \dots & b_{pn}
\end{pmatrix}_{n \times p}
$$

##### 3. Producto elemento a elemento $A \odot B^T$
Multiplicamos elemento a elemento.

$$
A \odot B^T = \begin{pmatrix}
a_{11}b_{11} & a_{12}b_{21} & \dots & a_{1p}b_{p1} \\
a_{21}b_{12} & a_{22}b_{22} & \dots & a_{2p}b_{p2} \\
\vdots & \vdots & \ddots & \vdots \\
a_{n1}b_{1n} & a_{n2}b_{2n} & \dots & a_{np}b_{pn}
\end{pmatrix}_{n \times p}
$$

##### 4. Reducción por Suma en el eje de columnas
Al sumar horizontalmente (sobre las $p$ columnas) para cada fila $i$, obtenemos:

$$\text{Fila}_i = a_{i1}b_{1i} + a_{i2}b_{2i} + \dots + a_{ip}b_{pi} = \sum_{k=1}^{p} a_{ik}b_{ki}$$

**Conclusión:** El resultado de la sumatoria es idéntico al valor $C_{ii}$ de la diagonal de la multiplicación original. Esto demuestra que se pueden obtener los valores de la diagonal de la multiplicacion de matrices, sin pasar por la multiplicacion en si. Este proceso es eficiente para $n$ muy grandes, permitiendonos operar sobre matrices de dimensiones $(n \times p)$ en lugar de $(n \times n)$.

$6$. Utilizar la propiedad antes demostrada para reimplementar la predicción del modelo `FasterQDA` de forma eficiente en un nuevo modelo `EfficientQDA`.

In [ ]:
class EfficientQDA(TensorizedQDA):
    def predict(self, X):
        unbiased_X = X[np.newaxis, :, :] - self.tensor_means # shape: (k, p, n)

        # (k, p, p) @ (k, p, n) -> (k, p, n)
        A = self.tensor_inv_cov @ unbiased_X

        # Aplicamos multiplicación elemento a elemento y sumamos por filas (axis=1)
        # unbiased_X actúa como B^T en la demostración
        quad_form = np.sum(unbiased_X * A, axis=1) # shape: (k, n)

        log_dets = np.log(LA.det(self.tensor_inv_cov)).reshape(-1, 1)
        log_conds = 0.5 * log_dets - 0.5 * quad_form

        log_posteriori = self.log_a_priori.reshape(-1, 1) + log_conds
        return np.argmax(log_posteriori, axis=0).reshape(1, -1)


In [ ]:
#EfficientQDA
model_EfficientQDA=EfficientQDA()
model_EfficientQDA.fit(X_train, y_train)
preds_EfficientQDA=model_EfficientQDA.predict(X_test)

#Comparo
if np.array_equal(preds_QDA, preds_EfficientQDA):
    print("Ambas predicciones son Iguales")

Ambas predicciones son Iguales


$7$. Comparar la performance de las 4 variantes de QDA implementadas hasta ahora (no Cholesky) ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

In [ ]:
# generamos el benchmark
b = Benchmark(
    X_letter, y_letter_encoded,
    n_runs = 50,
    warmup = 5,
    mem_runs = 50,
    test_sz = 0.3,
    same_splits = False
)

to_bench = [QDA, TensorizedQDA, FasterQDA, EfficientQDA]

for model in to_bench:
    b.bench(model)

summ = b.summary(baseline='QDA')

summ[["train_median_ms", "test_median_ms", "train_mem_median_mb", "test_mem_median_mb", "mean_accuracy"]]

Benching params:
Total runs: 105
Warmup runs: 5
Peak Memory usage runs: 50
Running time runs: 50
Train size rows (approx): 14000
Test size rows (approx): 6000
Test size fraction: 0.3


QDA (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/50 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/50 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/50 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/50 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,train_mem_median_mb,test_mem_median_mb,mean_accuracy
model,,,,,
QDA,9.60940,2984.79510,0.256035,0.254009,0.884467
TensorizedQDA,10.23750,481.73635,0.256004,0.169388,0.884480
FasterQDA,12.27295,2303.06595,0.256157,7179.315735,0.884703
EfficientQDA,11.17320,35.83935,0.256035,58.498512,0.884390


El analisis de los resultados revela una progresion en la optimización de los modelos: <br>
El modelo `QDA` original presenta el peor desempeño en tiempo debido a la ineficiencia de los ciclos `for`. De todas formas, mantiene el consumo de memoria mínimo al procesar los datos de forma secuencial. <br>
Al implementar `TensorizedQDA`, se logra una mejora sustancial en velocidad al vectorizar el cálculo sobre las clases $k$, manteniendo el consumo de memoria bajo. El bucle sobre las observaciones aun persiste.<br>
En la implementacion de `FasterQDA`, a pesar de intentar maximizar la velocidad (o minimizar el tiempo) eliminando todos los bucles `for` para procesar todas las observaciones juntas, el tiempo empleado y el consumo de memoria crecen de forma abrupta. Esto ocurre porque el modelo genera una matriz intermedia de $k × n × n$ que desborda la memoria y obliga al sistema a realizar transferencias masivas de datos. Se demuestra que, en este caso, eliminar el bucle `for` con una sola iteracion puede ser contraproducente.<br>
El modelo `EfficientQDA` representa la mejor optimización al alcanzar un tiempo de ejecución mucho mas bajo que el resto. Su éxito radica en calcular la diagonal del producto de matrices sin pasar por la matriz de $k × n × n$,  operando únicamente con tensores de tamaño $k × p × n$. El resultado es un modelo mucho más rápido que el original, manteniendo el consumo de memoria controlado.

Para verificar esta teoria sobre desbordamiento de la memoria en el `FasterQDA`, vamos a probar con un dataset mucho mas chico (penguins), donde calcular la matriz de $k × n × n$, no sea tan costosa por tener un $n$ significativamente menor

In [ ]:
X_penguins, y_penguins = get_penguins_dataset()

# encoding de labels
y_penguins_encoded = label_encode(y_penguins.reshape(-1,1))

b2 = Benchmark(
    X_penguins, y_penguins_encoded,
    n_runs = 100,
    warmup = 10,
    mem_runs = 100,
    test_sz = 0.3,
    same_splits = False
)

to_bench2 = [QDA, TensorizedQDA, FasterQDA, EfficientQDA]

for model in to_bench2:
    b2.bench(model)

summ2 = b2.summary(baseline='QDA')
summ2[["train_median_ms", "test_median_ms", "train_mem_median_mb", "test_mem_median_mb", "mean_accuracy"]]

Benching params:
Total runs: 210
Warmup runs: 10
Peak Memory usage runs: 100
Running time runs: 100
Train size rows (approx): 240
Test size rows (approx): 102
Test size fraction: 0.3


QDA (MEM):   0%|          | 0/100 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/100 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,train_mem_median_mb,test_mem_median_mb,mean_accuracy
model,,,,,
QDA,0.41255,5.73320,0.011604,0.004790,0.987184
TensorizedQDA,0.54305,2.71230,0.011513,0.004547,0.986699
FasterQDA,0.68335,0.13385,0.011421,0.264572,0.985146
EfficientQDA,0.53810,0.09550,0.011421,0.043373,0.986699


Se puede verificar que en este caso (con un $n$ chico dado el dataset) el modelo `FasterQDA` efectivamente es mas rapido que `TensorizedQDA`. Esto se debe a que el costo en memoria asociado al tensor de $k × n × n$ no es significativamente costoso para el nivel de memoria que posee el equipo. A medida que el numero de observaciones crece, el costo en memoria asociado crece de forma abrupta

$8$. Si una matriz $A$ tiene fact. de Cholesky $A=LL^T$, expresar $A^{-1}$ en términos de $L$. ¿Cómo podría esto ser útil en la forma cuadrática de QDA?

##### Expresión de $A^{-1}$ en términos de $L$:
Sea $A$ una matriz simétrica y definida positiva (condiciones que cumple cualquier matriz de covarianza), la descomposición de Cholesky establece que $A$ puede factorizarse como el producto de una matriz triangular inferior $L$ y su traspuesta $L^T$:
$$A = LL^T$$

Para encontrar $A^{-1}$, aplicamos la inversa a ambos lados de la ecuación:
$$A^{-1} = (LL^T)^{-1}$$

Por la propiedad de la inversa de un producto de matrices ($(XY)^{-1} = Y^{-1}X^{-1}$), invertimos el orden de los factores:
$$A^{-1} = (L^T)^{-1} L^{-1}$$

Por la propiedad que relaciona la inversa y la traspuesta ($(X^T)^{-1} = (X^{-1})^T$), podemos reescribir el primer término:
$$A^{-1} = (L^{-1})^T L^{-1}$$

**Conclusión:** La inversa de $A$ se puede calcular multiplicando la traspuesta de la inversa de $L$ por la inversa de $L$.

Utilidad en en la forma cuadratica de QDA:

En QDA necesitamos calcular una "Forma cuadrática" (Q) usando esta fórmula:
$$Q = x^T \cdot A^{-1} \cdot x$$
En este caso, $x$ es nuestro dato centrado respecto de la media ($x_c$ en algunas notaciones) y $A$ es la matriz de covarianza.

El problema es que calcular $A^{-1}$ es lento y propenso a errores de computacion. Podemos usar la factorización de Cholesky para esquivar este cálculo.

Sustituyendo $A$ por su factorización de Cholesky ($A = LL^T$) y aplicando el resultado obtenido anteriormente obtenemos:
$$Q = x^T \cdot \left( (L^{-1})^T L^{-1} \right) \cdot x$$

Por la propiedad asociativa de la multiplicación de matrices, agrupamos los términos:
$$Q = \left( x^T (L^{-1})^T \right) \cdot \left( L^{-1} x \right)$$

Notamos que el primer paréntesis es exactamente la traspuesta del segundo paréntesis, ya que $(YX)^T = X^T Y^T$. Por lo tanto:
$$Q = \left( L^{-1} x \right)^T \cdot \left( L^{-1} x \right)$$

Definimos un nuevo vector $y = L^{-1} x$. Reemplazando en la ecuación, la forma cuadrática se simplifica drásticamente al producto punto de un vector consigo mismo (la norma al cuadrado):
$$Q = y^T y = \|y\|^2$$

Por que es util esto?

Transformar el problema original en $Q = \|y\|^2$ donde $y = L^{-1} x$ ofrece ventajas rotundas:

1. Evita la Inversión Explícita (Estabilidad Numérica):
Calcular $A^{-1}$ directamente es numéricamente inestable si las características están altamente correlacionadas. Cholesky es mucho más robusto frente a errores de redondeo.

2. Resolución Eficiente de Sistemas Lineales:
El vector $y$ se define como $y = L^{-1} x$, lo cual es equivalente a resolver el sistema de ecuaciones lineales:
   $$L y = x$$
   Como $L$ es una matriz triangular inferior tiene todos ceros en la parte superior. Este sistema se resuelve mediante el algoritmo de *Sustitución Hacia Adelante* (Forward Substitution), ya que en la primer fila, todos los coeficientes son 0, menos el primer elemento.

3. Cálculo de Determinantes a "Costo Cero":
   La función de decisión de QDA también requiere calcular $\ln(|A|)$. Con Cholesky, el determinante de $A$ es simplemente el cuadrado del producto de los elementos de la diagonal de $L$:
   $$|A| = |L| |L^T| = \left( \prod_{i=1}^{p} L_{ii} \right)^2$$
   Esto transforma un cálculo complejo en una simple multiplicación de $p$ escalares y luego elevarlos al cuadrado.

**Conclusión:** Utilizar Cholesky nos permite calcular la forma cuadrática resolviendo un sistema triangular rápido y haciendo un producto punto, logrando un algoritmo más veloz, con menor consumo de recursos y matemáticamente más estable.

$9$. Explicar las diferencias entre `QDA_Chol1`y `QDA` y cómo `QDA_Chol1` llega, paso a paso, hasta las predicciones.

La principal diferencia radica en como realizan el calculo durante el `fit`

`QDA` estima la covarianza para cada clase $k$ y la invierte con `LA.inv()`. Lo que almacena es dicha matriz de convarianzas inversa: $\hat{\Sigma}_j^{-1}$

`QDA_Chol1` da un paso intermedio: primero factoriza $\hat{\Sigma}_j = L L^T$ mediante la descomposición de Cholesky y luego invierte $L$. Lo que almacena es $L^{-1}$:


Invertir $L$ (matriz triangular) es más estable numéricamente y más eficiente que invertir $\hat{\Sigma}_j$ directamente.

 Paso a paso de `QDA_Chol1`

Dado un punto $x$ y una clase $j$:

1. Centrar la observación:

$$x_c = x - \hat{\mu}_j$$

`unbiased_x = x - self.means[class_idx]`

2. Aplicar $L^{-1}$ al vector centrado:

$$y = L^{-1} \cdot x_c$$

`y = L_inv @ unbiased_x`

3. Calcular la forma cuadrática como norma al cuadrado:

Como se demostró en el punto anterior, la forma cuadrática se puede reescribir como:

$$Q = \|y\|^2  =  \sum_{i=1}^{p} y_i^2$$

`quad_form = (y**2).sum()`

4. Calcular el log-determinante desde la diagonal de $L^{-1}$:

El determinante de una matriz triangular es el producto de su diagonal, por lo que:

$$|\Sigma_j^{-1}| = |L^{-1}|^2 = \left(\prod_{i=1}^{p} (L^{-1})_{ii}\right)^2$$

$$\Rightarrow \quad \frac{1}{2}\log|\Sigma_j^{-1}| = \log \prod_{i=1}^{p} (L^{-1})_{ii}$$

`log_det = np.log(L_inv.diagonal().prod())`

En `QDA` se calculaba `0.5 * np.log(LA.det(inv_cov))`, que opera sobre la matriz completa. Aquí alcanza con multiplicar $p$ escalares.

5. Retornar la log-probabilidad condicional:

$$\log P(x \mid G = j) = \log \prod_{i}(L^{-1})_{ii} \;-\; \frac{1}{2}\|y\|^2$$

`return np.log(L_inv.diagonal().prod()) - 0.5 * (y**2).sum()`

Finalmente, `BaseBayesianClassifier` suma $\log \hat{\pi}_j$ (log-probabilidad a priori) para obtener la log-probabilidad a posteriori, y `_predict_one` devuelve la clase $j^*$ que la maximiza:

$$j^* = \underset{j}{\arg\max} \left\{ \log \hat{\pi}_j + \log P(x \mid G = j) \right\}$$

$10$. ¿Cuáles son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?

`QDA_Chol1`

Fit: calcula $L$ con Cholesky y luego invierte $L$ usando `LA.inv()`, un algoritmo de la libreria `NumPy`. Almacena $L^{-1}$.

Predict: obtiene $y$ como un producto usando la inversa ya precalculada:

$$y = L_j^{-1} \cdot x_c$$

Siendo $x_c$ el vector observacion centrado respecto de la media

Estrategia: Todo el trabajo pesado ocurre en el `fit`. La predicción es solo un producto matricial.

`QDA_Chol2`

Fit: calcula $L$ con Cholesky pero no la invierte. Almacena $L$ directamente.

Predict: en lugar de multiplicar por $L^{-1}$, resuelve el sistema triangular $L \cdot y = x_c$ en cada predicción mediante sustitución hacia adelante (forward substitution): `y = solve_triangular(L, unbiased_x, lower=True)`

Ambas operaciones son algebraicamente equivalentes: resolver $Ly = x_c$ produce el mismo $y$ que calcular $L^{-1}x_c$.

Estrategia: El fit es el más liviano de los tres. El costo se paga en cada llamada al predict ya que se debe resolver el sistema de ecuaciones. Si bien es el mas costoso en la prediccion, presenta una ventaja clave: al no invertir nunca ninguna matriz, es el más estable numéricamente. Si $\hat{\Sigma}_j$ está cerca de ser singular, `LA.inv()` amplifica errores de redondeo, mientras que `solve_triangular` los contiene mejor.

`QDA_Chol3`

Fit: calcula $L$ con Cholesky y la invierte usando `dtrtri`, una rutina de LAPACK diseñada específicamente para invertir matrices triangulares. Esta rutina es de super bajo nivel y esta escrita en C/Fortran. Almacena $L^{-1}$.

Predict: Funciona igual que `QDA_Chol1`

Estrategia: La misma que `QDA_Chol1`. La diferencia se da en el `fit`. Llama a `dtrtri` que utiliza la estructura triangular de $L$ para invertirla, mientras que `LA.inv()` aplica un algoritmo que no asume ninguna estructura especial.


$11$. Comparar la performance de las 7 variantes de QDA implementadas hasta ahora ¿Qué se observa?¿Hay alguna de las implementaciones de `QDA_Chol` que sea claramente mejor que las demás?¿Alguna que sea peor?

In [ ]:
to_bench = [QDA_Chol1, QDA_Chol2, QDA_Chol3]

for model in to_bench:
    b.bench(model)

summ = b.summary(baseline='QDA')
summ[["train_median_ms", "test_median_ms", "train_mem_median_mb", "test_mem_median_mb", "mean_accuracy"]]

QDA_Chol1 (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

QDA_Chol1 (TIME):   0%|          | 0/50 [00:00<?, ?it/s]

QDA_Chol2 (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/50 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/50 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,train_mem_median_mb,test_mem_median_mb,mean_accuracy
model,,,,,
QDA,9.60940,2984.79510,0.256035,0.254009,0.884467
TensorizedQDA,10.23750,481.73635,0.256004,0.169388,0.884480
FasterQDA,12.27295,2303.06595,0.256157,7179.315735,0.884703
EfficientQDA,11.17320,35.83935,0.256035,58.498512,0.884390
QDA_Chol1,10.60175,1781.07015,0.255791,0.110233,0.883723
QDA_Chol2,9.64265,4416.21645,0.256004,0.251442,0.885293
QDA_Chol3,10.18575,1831.23215,0.255974,0.110706,0.883963


`QDA_Chol2` es mas lenta que el resto porque en el `predict` debe resolver un sistema de ecuaciones lineales. Este proceso tarda mas que realizar una multiplicacion de matrices. `QDA_Chol1` y `QDA_Chol3` estan equilibradas en velocidad y consumo de memoria. Cabe mencionar, que todos los modelos, todavia siguen iterando clase por clase y observacion por observacion

$12$. Implementar el modelo `TensorizedChol` paralelizando sobre clases/observaciones según corresponda. Se recomienda heredarlo de alguna de las implementaciones de `QDA_Chol`, aunque la elección de cuál de ellas queda a cargo del alumno según lo observado en los benchmarks de puntos anteriores.

In [ ]:
class TensorizedChol(QDA_Chol1):
    def _fit_params(self, X, y):
        super()._fit_params(X, y)
        # Agrupamos en tensores las matrices inversas triangulares y las medias
        self.tensor_L_invs = np.stack(self.L_invs) # (k, p, p)
        self.tensor_means = np.stack(self.means) # (k, p, 1)

    def _predict_log_conditionals(self, x):
        unbiased_x = x - self.tensor_means # (k, p, 1)

        # y_chol = L^{-1} @ (x - mu)
        y_chol = self.tensor_L_invs @ unbiased_x # (k, p, p) @ (k, p, 1) -> (k, p, 1)

        # Sumamos los cuadrados a lo largo de las features
        quad_form = np.sum(y_chol**2, axis=(1, 2)) # (k,)

        # Determinante
        diags = np.diagonal(self.tensor_L_invs, axis1=1, axis2=2)
        log_dets = np.log(np.prod(diags, axis=1)) # (k,)

        return log_dets - 0.5 * quad_form

    def _predict_one(self, x):
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

In [ ]:
#TensorizedChol
model_TensorizedChol=TensorizedChol()
model_TensorizedChol.fit(X_train, y_train)
preds_TensorizedChol=model_TensorizedChol.predict(X_test)

#Comparo
if np.array_equal(preds_QDA, preds_TensorizedChol):
    print("Ambas predicciones son Iguales")

Ambas predicciones son Iguales


$13$. Implementar el modelo `EfficientChol` combinando los insights de `EfficientQDA` y `TensorizedChol`. Si se desea, se puede implementar `FasterChol` como ayuda, pero no se contempla para el punto.

In [ ]:
class EfficientChol(TensorizedChol):
    def predict(self, X):
        unbiased_X = X[np.newaxis, :, :] - self.tensor_means # (k, p, n)

        # Batch tensor multiplication
        Z = self.tensor_L_invs @ unbiased_X # (k, p, p) @ (k, p, n) -> (k, p, n)

        # Como es una matriz que ya contiene a L^{-1} x, solo falta elevar al cuadrado
        # y sumar sobre el axis de las variables (axis=1).
        quad_form = np.sum(Z**2, axis=1) # (k, n)

        diags = np.diagonal(self.tensor_L_invs, axis1=1, axis2=2) # (k, p)
        log_dets = np.log(np.prod(diags, axis=1)).reshape(-1, 1) # (k, 1)

        log_conds = log_dets - 0.5 * quad_form
        log_posteriori = self.log_a_priori.reshape(-1, 1) + log_conds

        return np.argmax(log_posteriori, axis=0).reshape(1, -1)

In [ ]:
#EfficientChol
model_EfficientChol=EfficientChol()
model_EfficientChol.fit(X_train, y_train)
preds_EfficientChol=model_EfficientChol.predict(X_test)

#Comparo
if np.array_equal(preds_QDA, preds_EfficientChol):
    print("Ambas predicciones son Iguales")

Ambas predicciones son Iguales


$14$. Comparar la performance de las 9 variantes de QDA implementadas ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

In [ ]:
to_bench = [TensorizedChol, EfficientChol]

for model in to_bench:
    b.bench(model)

summ = b.summary(baseline='QDA')
summ[["train_median_ms", "test_median_ms", "train_mem_median_mb", "test_mem_median_mb", "mean_accuracy"]]

TensorizedChol (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

TensorizedChol (TIME):   0%|          | 0/50 [00:00<?, ?it/s]

EfficientChol (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/50 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,train_mem_median_mb,test_mem_median_mb,mean_accuracy
model,,,,,
QDA,9.60940,2984.79510,0.256035,0.254009,0.884467
TensorizedQDA,10.23750,481.73635,0.256004,0.169388,0.884480
FasterQDA,12.27295,2303.06595,0.256157,7179.315735,0.884703
EfficientQDA,11.17320,35.83935,0.256035,58.498512,0.884390
QDA_Chol1,10.60175,1781.07015,0.255791,0.110233,0.883723
QDA_Chol2,9.64265,4416.21645,0.256004,0.251442,0.885293
QDA_Chol3,10.18575,1831.23215,0.255974,0.110706,0.883963
TensorizedChol,11.94750,241.54860,0.256035,0.176270,0.884480
EfficientChol,12.48600,28.84785,0.255852,58.498512,0.884510


Al observar los tiempos de ejecución y los consumos de memoria de las 9 variantes se observa: <br>
Los modelos con bucles sobre clases (`QDA`, `QDA_Chol1`, `QDA_Chol2`, `QDA_Chol3`): Son los más lentos en la fase de predicción (test_mean_ms) porque utilizan dos ciclos `for`. Uno iterando sobre las $k$ clases y otro iterando sobre las $n$ observaciones

Las vectorizaciones (`TensorizedQDA`, `FasterQDA` y `TensorizedChol`): Mejoran el tiempo computacional eliminando el bucle sobre las $k$ clases. Sin embargo, `FasterQDA` (y su equivalente en Cholesky si se hubiera implementado) intentan vectorizar sobre las $n$ observaciones usando un producto interno directo que genera un tensor intermedio gigantesco de tamaño $(k,n,n)$. Esto causa un pico en el uso de memoria que los hace inviables en datasets grandes.

Las vectorizaciones eficientes (`EfficientQDA`, `EfficientChol`): Son las variantes ganadoras absolutas en tiempo y memoria durante la predicción. Al esquivar la necesidad de una matriz gigantesca, logran predecir todas las observaciones simultáneamente en bloque, evitando los bucles `for`

Los resultados obtenidos se alinean de manera perfecta con la teoría. Se puede observar como aplicar la teoria nos ayuda a reducir tiempos de ejecucion

En conclusión, `EfficientChol` es la mejor implementacion, ya que junta la estabilidad algebraica de la descomposición de Cholesky con las ventajas de la vectorización en bloque

### Bonus - Comparacion contra SKLearn

Vamos a probar estos modelos contra `QuadraticDiscriminantAnalysis` que trae la libreria `Scikit-Learn` <br>
Primero vamos a crear un `Wrapper` ya que la forma de los datos de input y output son distintos para `Scikit-Learn`. En los modelos que trabajamos hasta aca, cada fila es un predictor, y cada columna una observacion. `Scikit-Learn` espera los datos justamente al reves, cada fila una observacion y cada columna un predictor. Para poder usar el `benchmark` y comparar los resultados, necesitamos pasarle el mismo input a todos los modelos

In [ ]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

class SklearnQDAWrapper:
    """
    Wrapper de QuadraticDiscriminantAnalysis de sklearn compatible con el Benchmark.

    El Benchmark maneja los datos en formato columnas:
        X : (p, n)  — cada columna es una observación
        y : (1, n)  — fila única de etiquetas

    Sklearn espera el formato
        X : (n, p) — cada columna es un predictor
        y : (n,)
    Este wrapper realiza las conversiones necesarias en fit y predict.
    """

    def __init__(self, **kwargs):
        self._model = QuadraticDiscriminantAnalysis(**kwargs)

    def fit(self, X, y, a_priori=None):
        """
        X : (p, n)  →  se transpone a (n, p) para sklearn
        y : (1, n)  →  se aplana a (n,)   para sklearn
        a_priori se ignora (sklearn lo maneja internamente con priors=)
        """
        self._model.fit(X.T, y.flatten())
        return self

    def predict(self, X):
        """
        X : (p, n)  →  se transpone a (n, p) para sklearn
        salida : (1, n)  para que sea compatible con el Benchmark
        """
        return self._model.predict(X.T).reshape(1, -1)

In [ ]:
# SklearnQDAWrapper
model_sklearn = SklearnQDAWrapper()
model_sklearn.fit(X_train, y_train)
preds_sklearn = model_sklearn.predict(X_test)

if np.array_equal(preds_QDA, preds_sklearn):
    print("Ambas predicciones son Iguales")
else:
    n_diff = (preds_QDA != preds_sklearn).sum()
    print(f"Difieren en {n_diff} observaciones de {preds_QDA.size}")

Difieren en 1 observaciones de 6000


Podemos observar que las prediciones no son exactamente igual para todos los casos. Para 1 caso de 6000, estima otro valor. Podemos tolerar este margen de error

In [ ]:
# Benchmark
to_bench = [SklearnQDAWrapper]

for model in to_bench:
    b.bench(model)

summ = b.summary(baseline='QDA')

summ[["train_median_ms", "test_median_ms", "train_mem_median_mb", "test_mem_median_mb", "mean_accuracy"]]

SklearnQDAWrapper (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

SklearnQDAWrapper (TIME):   0%|          | 0/50 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,train_mem_median_mb,test_mem_median_mb,mean_accuracy
model,,,,,
QDA,9.60940,2984.79510,0.256035,0.254009,0.884467
TensorizedQDA,10.23750,481.73635,0.256004,0.169388,0.884480
FasterQDA,12.27295,2303.06595,0.256157,7179.315735,0.884703
EfficientQDA,11.17320,35.83935,0.256035,58.498512,0.884390
QDA_Chol1,10.60175,1781.07015,0.255791,0.110233,0.883723
QDA_Chol2,9.64265,4416.21645,0.256004,0.251442,0.885293
QDA_Chol3,10.18575,1831.23215,0.255974,0.110706,0.883963
TensorizedChol,11.94750,241.54860,0.256035,0.176270,0.884480
EfficientChol,12.48600,28.84785,0.255852,58.498512,0.884510


Al contrastar nuestra implementación optimizada `EfficientChol` con el modelo estándar de Scikit-Learn `QuadraticDiscriminantAnalysis`, se desprenden las siguientes conclusiones:
1. Eficiencia en el Entrenamiento: <br>
Ambas implementaciones presentan tiempos de ejecución prácticamente idénticos en la etapa de entrenamiento. Esto es razonable, ya que el cálculo de los vectores de medias y las matrices de covarianza por clase son una operación estándar. <br>
Sin embargo, observamos que `Scikit-Learn` consume una mayor cantidad de memoria en entrenamiento. Esto sugiere que la librería realiza el calculo y almacenamiento de estructuras auxiliares para optimizar la fase posterior de predicción.
2. Rendimiento en el Test/Inferencia: <br>
En la etapa de predicción, `Scikit-Learn` demuestra una optimización superior tanto en tiempo como en memoria. Esta diferencia es el resultado directo de la inversión realizada durante el entrenamiento. <br>

A pesar de la ventaja técnica de una librería como `Scikit-Learn`, los resultados obtenidos por nuestro modelo `EfficientChol` son más que satisfactorios